# Closing line value (CLV): grade your bets against the close

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JacobiusMakes/parlayapi-notebooks/blob/main/04-closing-line-value.ipynb)

The closing line is the last price before a game starts, and it is the sharpest
number the market produces: every model, injury report, and dollar of action is
already in it. Beating the price you bet at versus the close, consistently, is the
most widely accepted evidence that a bettor has real edge. This notebook:

1. implements CLV math (price vs close, and EV vs the devigged close),
2. shows the tier lookback windows honestly, fetched live from `/pricing`,
3. pulls real closing lines from the historical archive (**API key required** for
   this part) and grades example bets against them.

The archive behind this reaches back to 2005 and includes sharp-book (Pinnacle)
closes, which is the benchmark most CLV work uses.

In [1]:
# ---- Config: paste your API key between the quotes ----
# Get a free key at https://parlay-api.com/signup (free tier, no card required).
# Leave it empty to run against the keyless demo endpoint instead.
API_KEY = ""

import os
API_KEY = (API_KEY or os.environ.get("PARLAYAPI_KEY", "")).strip()
BASE_URL = "https://parlay-api.com"
SPORT = "baseball_mlb"  # also try: basketball_nba, americanfootball_nfl, icehockey_nhl, soccer_epl

if API_KEY:
    print("API key set: using the full keyed endpoints.")
else:
    print("No API key set: falling back to the keyless demo endpoint.")
    print("The demo serves the first 5 events per sport, moneyline (h2h) only,")
    print("capped at 60 requests per hour. Paste a free key above for all events,")
    print("all 30+ books, and every market:", "https://parlay-api.com/signup")

No API key set: falling back to the keyless demo endpoint.
The demo serves the first 5 events per sport, moneyline (h2h) only,
capped at 60 requests per hour. Paste a free key above for all events,
all 30+ books, and every market: https://parlay-api.com/signup


In [2]:
# American <-> decimal conversions, matching the conventions used by
# https://parlay-api.com/tools/no-vig-calculator and /tools/parlay-calculator.

def american_to_decimal(a):
    """+150 -> 2.5, -110 -> 1.9091. Valid American odds are >= +100 or <= -100."""
    a = float(a)
    if abs(a) < 100:
        raise ValueError(f"{a} is not a valid American price (must be >= +100 or <= -100)")
    if a > 0:
        return 1 + a / 100
    return 1 + 100 / (-a)

def decimal_to_american(d):
    """2.5 -> +150, 1.9091 -> -110 (rounded to the nearest integer)."""
    d = float(d)
    if d <= 1:
        raise ValueError(f"decimal odds must be > 1, got {d}")
    if d >= 2:
        return round((d - 1) * 100)
    return round(-100 / (d - 1))

def implied_prob(decimal_odds):
    """Implied win probability of decimal odds (includes the vig)."""
    return 1.0 / float(decimal_odds)

def fmt_american(a):
    return ("+" if a > 0 else "") + str(int(a))

## CLV math

Two complementary ways to grade a bet price against the close:

- **Price CLV**: compare decimal odds directly.
  `clv_pct = (your_dec / close_dec - 1) * 100`. Positive means you beat the close.
- **EV vs the devigged close**: strip the vig from the closing market
  (proportional method, as in notebook 02), take the fair closing probability of
  your side, and compute `EV = p_close * (your_dec - 1) - (1 - p_close)`. This is
  the better measure, because it grades you against the fair close instead of a
  vigged price.

In [3]:
def devig_proportional_probs(american_prices):
    raw = [1.0 / american_to_decimal(a) for a in american_prices]
    overround = sum(raw)
    return [r / overround for r in raw]

def price_clv_pct(your_price, close_price):
    """Positive = you beat the closing price."""
    return (american_to_decimal(your_price) / american_to_decimal(close_price) - 1) * 100

def ev_vs_fair_close(your_price, close_market_prices, your_index=0):
    """EV per $1 of your price against the devigged closing market.

    close_market_prices: all sides of the closing market (e.g. [home_ml, away_ml]).
    your_index: which side you bet.
    """
    p_close = devig_proportional_probs(close_market_prices)[your_index]
    dec = american_to_decimal(your_price)
    return p_close * (dec - 1) - (1 - p_close)

# Worked example (hypothetical prices, labeled as such):
# You bet the home side at -105 on Tuesday; it closes -120 / +100.
your_price = -105
close = [-120, +100]
print(f"you bet {fmt_american(your_price)}, market closed "
      f"{fmt_american(close[0])} / {fmt_american(close[1])}")
print(f"  price CLV vs closing side: {price_clv_pct(your_price, close[0]):+.2f}%")
print(f"  fair close prob of your side: {devig_proportional_probs(close)[0]:.4f}")
print(f"  EV vs fair close: {ev_vs_fair_close(your_price, close)*100:+.2f}% of stake")

you bet -105, market closed -120 / +100
  price CLV vs closing side: +6.49%
  fair close prob of your side: 0.5217
  EV vs fair close: +1.86% of stake


## How much history can you see?

Lookback depth is a plan feature. Rather than hardcode numbers that go stale, ask
the API: `/pricing` returns the current tiers as JSON, including each tier's
`historical_data_hours`. Current plans and prices live at
[parlay-api.com/pricing](https://parlay-api.com/pricing); the free tier is enough
to run this notebook against the last couple of days.

In [4]:
import requests
import pandas as pd

try:
    r = requests.get(f"{BASE_URL}/pricing", timeout=30)
    tiers = r.json()["parlay_api"]
    lookback = pd.DataFrame(
        [{"tier": t["tier"],
          "historical_lookback_hours": t["historical_data_hours"],
          "historical_lookback_days": round(t["historical_data_hours"] / 24, 1)}
         for t in tiers])
    display(lookback)
    print("Full plan details: https://parlay-api.com/pricing")
except Exception:
    print("Could not parse /pricing right now; see https://parlay-api.com/pricing")

,tier,historical_lookback_hours,historical_lookback_days
0,free,48,2.0
1,starter,168,7.0
2,pro,720,30.0
3,business,2160,90.0
4,enterprise,8760,365.0
5,scale,87600,3650.0


Full plan details: https://parlay-api.com/pricing


## Pull real closing lines (API key required)

`GET /v1/historical/sports/{sport}/closing-odds` returns flat rows: one row per
(game, bookmaker) with `home_odds` / `away_odds` (and `draw_odds` where the sport
has draws), final scores, and the result. Game-line queries default to the
Pinnacle close, which is exactly what you want for CLV grading. The call costs 10
API credits at the time of writing (see [the docs](https://parlay-api.com/docs)).

If you leave the date parameters off, the API automatically serves the newest
rows inside your tier's lookback window, so this cell works on every tier.
Without a key it prints a pointer and moves on; the math cells above already ran.

In [5]:
closes = pd.DataFrame()
if not API_KEY:
    print("No API key set, so skipping the live archive pull.")
    print("Grab a free key at https://parlay-api.com/signup and paste it in the")
    print("config cell at the top to run this section against real closing lines.")
else:
    r = requests.get(
        f"{BASE_URL}/v1/historical/sports/{SPORT}/closing-odds",
        params={"markets": "h2h", "bookmakers": "pinnacle", "limit": 200},
        headers={"X-API-Key": API_KEY},
        timeout=60,
    )
    if r.ok:
        closes = pd.DataFrame(r.json())
        if closes.empty:
            print(f"No closing rows for {SPORT} inside your tier window. Try another")
            print("sport key, or a paid tier for deeper lookback (see /pricing).")
        else:
            cols = [c for c in ["game_date", "home_team", "away_team", "bookmaker",
                                "home_odds", "away_odds", "draw_odds",
                                "home_score", "away_score", "result"] if c in closes.columns]
            print(f"{len(closes)} closing rows, newest game_date {closes['game_date'].max()}")
            display(closes[cols].head(8))
    else:
        print(f"Archive call failed: HTTP {r.status_code}: {r.text[:200]}")

No API key set, so skipping the live archive pull.
Grab a free key at https://parlay-api.com/signup and paste it in the
config cell at the top to run this section against real closing lines.


## Grade bets against the real close

To grade a real bet you would join on the game and side. Here we grade a
what-if: suppose you had bet every home side at today's typical -110 shelf price;
how does that price stand against each game's actual Pinnacle close? Games where
the home side closed shorter than -110 graded positive, games where it closed
longer graded negative. Swap in your own bet log (game, side, price) to make this
real.

In [6]:
if closes.empty:
    print("No archive rows loaded above, so nothing to grade here.")
else:
    graded = closes.dropna(subset=["home_odds", "away_odds"]).copy()
    if graded.empty:
        print("Rows loaded, but none carried a two-sided h2h close to grade against.")
    else:
        HYPOTHETICAL_BET_PRICE = -110  # stand-in for your actual bet price
        graded["clv_pct"] = [
            price_clv_pct(HYPOTHETICAL_BET_PRICE, h) for h in graded["home_odds"]
        ]
        graded["ev_vs_fair_close_pct"] = [
            100 * ev_vs_fair_close(HYPOTHETICAL_BET_PRICE, [h, a])
            for h, a in zip(graded["home_odds"], graded["away_odds"])
        ]
        cols = ["game_date", "home_team", "away_team",
                "home_odds", "clv_pct", "ev_vs_fair_close_pct"]
        print(f"grading a hypothetical {fmt_american(HYPOTHETICAL_BET_PRICE)} home bet "
              f"against {len(graded)} real closes")
        display(graded[cols].sort_values("clv_pct", ascending=False)
                .head(10).round(2))
        print(f"mean EV vs fair close: {graded['ev_vs_fair_close_pct'].mean():+.2f}%")

No archive rows loaded above, so nothing to grade here.


## Notes for real use

- Beating the close on a handful of bets means little; CLV is a long-run signal.
- Devig the close before grading (as done here). Grading against the vigged
  closing price flatters every bet by roughly half the vig.
- Draw sports (soccer) need the three-way devig: pass
  `[home_odds, draw_odds, away_odds]` to `ev_vs_fair_close` and pick your index.
- Lookback depth is tiered (see the live table above and
  [parlay-api.com/pricing](https://parlay-api.com/pricing)); the archive itself
  reaches back to 2005.
- ParlayAPI also has a hosted CLV grader (`/v1/clv`) if you would rather POST your
  bets than run the math client-side; see [the docs](https://parlay-api.com/docs).

---

**More ParlayAPI resources**

- Docs: [parlay-api.com/docs](https://parlay-api.com/docs)
- Free API key (no card): [parlay-api.com/signup](https://parlay-api.com/signup)
- Browser calculators the math here matches: [no-vig](https://parlay-api.com/tools/no-vig-calculator), [parlay](https://parlay-api.com/tools/parlay-calculator), [EV](https://parlay-api.com/tools/ev-calculator)
- The rest of this series: [github.com/JacobiusMakes/parlayapi-notebooks](https://github.com/JacobiusMakes/parlayapi-notebooks)

These notebooks are for research and education. Nothing here is betting advice.